# TabNet to ONNX Conversion

This notebook converts a trained TabNet model and StandardScaler to ONNX format for optimized inference.

**Date:** 2025-09-30

---

## Overview

Converts TabNet PyTorch model to ONNX format for production deployment.

**Input Files:**
- `model.zip` - Trained TabNet model (PyTorch format)
- `scaler.joblib` - StandardScaler for preprocessing

**Output Files:**
- `onnx_models/model.onnx` - TabNet model in ONNX format
- `onnx_models/scaler.onnx` - StandardScaler in ONNX format
- `onnx_models/onnx_metadata.json` - Model metadata


## Imports and Setup

In [ ]:
import json
import warnings
import zipfile
from pathlib import Path
from typing import Dict, Tuple, Optional

import joblib
import numpy as np
import torch
import onnx
import onnxruntime as rt
from skl2onnx import convert_sklearn
from skl2onnx.common.data_types import FloatTensorType

warnings.filterwarnings('ignore')

print("All imports successful")

## Configuration

In [ ]:
# File paths
tabnet_dir = Path(".").resolve()
model_zip_path = tabnet_dir / "model.zip"
scaler_path = tabnet_dir / "scaler.joblib"
output_dir = tabnet_dir / "onnx_models"

# Create output directory
output_dir.mkdir(parents=True, exist_ok=True)

print(f"Working directory: {tabnet_dir}")
print(f"Output directory: {output_dir}")
print(f"\nInput files:")
print(f"  Model: {model_zip_path.name}")
print(f"  Scaler: {scaler_path.name}")

## Load TabNet Model

In [ ]:
print("Loading TabNet model...\n")

# Check if model file exists
if not model_zip_path.exists():
    raise FileNotFoundError(f"Model not found: {model_zip_path}")

# Extract model from zip
import tempfile
temp_dir = tempfile.mkdtemp()
with zipfile.ZipFile(model_zip_path, 'r') as zip_ref:
    zip_ref.extractall(temp_dir)

# Load TabNet model
# Note: Adjust loading method based on how TabNet model was saved
model_path = Path(temp_dir)
try:
    # Try loading as PyTorch model
    model = torch.load(model_path / 'model.pt', map_location='cpu')
    model.eval()
    print(f"Model loaded: {type(model).__name__}")
except Exception as e:
    print(f"Error loading model: {e}")
    print("\nModel files in archive:")
    for file in model_path.rglob('*'):
        if file.is_file():
            print(f"  {file.relative_to(model_path)}")
    raise

# Get model input shape
# This depends on your TabNet implementation
n_features = model.input_dim if hasattr(model, 'input_dim') else 78
print(f"Number of features: {n_features}")

## Load StandardScaler

In [ ]:
print("\nLoading StandardScaler...\n")

# Load scaler
if not scaler_path.exists():
    raise FileNotFoundError(f"Scaler not found: {scaler_path}")

scaler = joblib.load(scaler_path)
print(f"Scaler loaded: {type(scaler).__name__}")
print(f"Scaler features: {scaler.n_features_in_}")

if scaler.n_features_in_ != n_features:
    warnings.warn(f"Feature mismatch: model={n_features}, scaler={scaler.n_features_in_}")

## Convert StandardScaler to ONNX

In [ ]:
print("Converting StandardScaler to ONNX...\n")

scaler_onnx_path = output_dir / "scaler.onnx"

# Define input types
initial_types = [
    ('float_input', FloatTensorType([None, n_features]))
]

# Convert
onnx_scaler = convert_sklearn(
    scaler,
    initial_types=initial_types,
    target_opset=12
)

# Save
with open(scaler_onnx_path, "wb") as f:
    f.write(onnx_scaler.SerializeToString())

print(f"Scaler converted and saved to: {scaler_onnx_path.name}")

# Validate
onnx_model = onnx.load(scaler_onnx_path)
onnx.checker.check_model(onnx_model)
print(f"ONNX model validation passed")

# Check size
scaler_size = scaler_onnx_path.stat().st_size / 1024
print(f"Model size: {scaler_size:.2f} KB")

## Convert TabNet Model to ONNX

In [ ]:
print("\nConverting TabNet model to ONNX...\n")

model_onnx_path = output_dir / "model.onnx"

# Create dummy input for tracing
dummy_input = torch.randn(1, n_features, dtype=torch.float32)

# Export to ONNX using torch.onnx.export
torch.onnx.export(
    model,
    dummy_input,
    model_onnx_path,
    export_params=True,
    opset_version=12,
    do_constant_folding=True,
    input_names=['input'],
    output_names=['output'],
    dynamic_axes={
        'input': {0: 'batch_size'},
        'output': {0: 'batch_size'}
    }
)

print(f"Model converted and saved to: {model_onnx_path.name}")

# Validate
onnx_model_check = onnx.load(model_onnx_path)
onnx.checker.check_model(onnx_model_check)
print(f"ONNX model validation passed")

# Check size
model_size = model_onnx_path.stat().st_size / (1024 * 1024)
print(f"Model size: {model_size:.2f} MB")

## Save ONNX Metadata

In [ ]:
print("\nSaving ONNX metadata...\n")

metadata_onnx_path = output_dir / "onnx_metadata.json"

onnx_metadata = {
    'model_type': 'TabNet',
    'conversion_date': '2025-09-30',
    'n_features': n_features,
    'opset_version': 12,
    'onnx_runtime_version': rt.__version__,
    'torch_version': torch.__version__,
    'input_shape': [None, n_features],
    'input_dtype': 'float32',
    'output_labels': [0, 1],
    'notes': 'Use scaler.onnx first, then model.onnx for predictions'
}

with open(metadata_onnx_path, 'w') as f:
    json.dump(onnx_metadata, f, indent=2)

print(f"Metadata saved to: {metadata_onnx_path.name}")

## Verify Conversion with Test Inference

In [ ]:
print("\nVerifying conversion with test inference...\n")

# Create test data
test_data = np.random.randn(5, n_features).astype(np.float32)

# Original PyTorch model
with torch.no_grad():
    torch_output = model(torch.from_numpy(test_data))
    if isinstance(torch_output, tuple):
        torch_output = torch_output[0]
    torch_predictions = torch_output.numpy()

print("PyTorch predictions:")
print(torch_predictions[:3])

# ONNX model
onnx_session = rt.InferenceSession(str(model_onnx_path))
input_name = onnx_session.get_inputs()[0].name
onnx_output = onnx_session.run(None, {input_name: test_data})[0]

print("\nONNX predictions:")
print(onnx_output[:3])

# Compare
max_diff = np.max(np.abs(torch_predictions - onnx_output))
print(f"\nMaximum difference: {max_diff:.2e}")
print(f"Conversion {'successful' if max_diff < 1e-5 else 'may have issues'}")

## Conversion Summary

In [ ]:
print("\n" + "="*70)
print("CONVERSION COMPLETE")
print("="*70)
print(f"\nOutput directory: {output_dir}\n")
print(f"Generated files:")
print(f"  {scaler_onnx_path.name} ({scaler_size:.2f} KB)")
print(f"  {model_onnx_path.name} ({model_size:.2f} MB)")
print(f"  {metadata_onnx_path.name}")
print(f"\nNext step: Run notebook 02_test_tabnet_onnx.ipynb to validate conversion")